In [5]:
import json
from pymongo import MongoClient, InsertOne
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["yelp_db"]

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_DIR = r"D:\Projects\DSM\Assignment2\Yelp JSON\yelp_dataset"
REVIEW_LIMIT  = 200_000   # adjust if memory is tight
USER_LIMIT    = 100_000
CHECKIN_LIMIT = 50_000
TIP_LIMIT     = 50_000
BATCH         = 5_000
# ────────────────────────────────────────────────────────────────────────────

def load_jsonl(filepath, limit=None, transform=None):
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            obj = json.loads(line)
            if transform:
                obj = transform(obj)
            records.append(obj)
    return records

def bulk_insert(collection, records, batch_size=BATCH):
    for i in range(0, len(records), batch_size):
        batch = records[i:i+batch_size]
        ops = [InsertOne(r) for r in batch]
        collection.bulk_write(ops, ordered=False)
    print(f"  Inserted {len(records)} docs into '{collection.name}'")

# ── BUSINESSES ───────────────────────────────────────────────────────────────
print("Loading businesses...")
def transform_business(b):
    b["_id"] = b.pop("business_id")
    if isinstance(b.get("categories"), str):
        b["categories"] = [c.strip() for c in b["categories"].split(",")]
    return b

businesses = load_jsonl(f"{DATA_DIR}\\yelp_academic_dataset_business.json",
                        transform=transform_business)
db.businesses.drop()
bulk_insert(db.businesses, businesses)

# ── USERS ────────────────────────────────────────────────────────────────────
print("Loading users (subset)...")
def transform_user(u):
    u["_id"] = u.pop("user_id")
    if isinstance(u.get("friends"), str) and u["friends"]:
        u["friends"] = [x.strip() for x in u["friends"].split(",")]
    else:
        u["friends"] = []
    if isinstance(u.get("elite"), str) and u["elite"]:
        u["elite"] = [int(x.strip()) for x in u["elite"].split(",") if x.strip().isdigit()]
    return u

users = load_jsonl(f"{DATA_DIR}\\yelp_academic_dataset_user.json",
                   limit=USER_LIMIT, transform=transform_user)
db.users.drop()
bulk_insert(db.users, users)

# ── REVIEWS ──────────────────────────────────────────────────────────────────
print("Loading reviews (subset)...")
def transform_review(r):
    r["_id"] = r.pop("review_id")
    r["date"] = datetime.strptime(r["date"], "%Y-%m-%d %H:%M:%S")
    return r

reviews = load_jsonl(f"{DATA_DIR}\\yelp_academic_dataset_review.json",
                     limit=REVIEW_LIMIT, transform=transform_review)
db.reviews.drop()
bulk_insert(db.reviews, reviews)

# ── CHECKINS ─────────────────────────────────────────────────────────────────
print("Loading checkins (subset)...")
checkins = load_jsonl(f"{DATA_DIR}\\yelp_academic_dataset_checkin.json",
                      limit=CHECKIN_LIMIT)
db.checkins.drop()
bulk_insert(db.checkins, checkins)

# ── TIPS ─────────────────────────────────────────────────────────────────────
print("Loading tips (subset)...")
tips = load_jsonl(f"{DATA_DIR}\\yelp_academic_dataset_tip.json",
                  limit=TIP_LIMIT)
db.tips.drop()
bulk_insert(db.tips, tips)

# ── INDEXES ──────────────────────────────────────────────────────────────────
print("Creating indexes...")
db.reviews.create_index("business_id")
db.reviews.create_index("user_id")
db.reviews.create_index("date")
db.reviews.create_index([("business_id", 1), ("date", 1)])
db.businesses.create_index("city")
db.businesses.create_index("categories")
db.businesses.create_index([("city", 1), ("categories", 1)])
db.checkins.create_index("business_id")

print("Done.")

Loading businesses...
  Inserted 150346 docs into 'businesses'
Loading users (subset)...
  Inserted 100000 docs into 'users'
Loading reviews (subset)...
  Inserted 200000 docs into 'reviews'
Loading checkins (subset)...
  Inserted 50000 docs into 'checkins'
Loading tips (subset)...
  Inserted 50000 docs into 'tips'
Creating indexes...
Done.


In [6]:
# queries_mongo.py
from pymongo import MongoClient
import json
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["yelp_db"]

def pprint(label, result):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print('='*60)
    if isinstance(result, list):
        for r in result[:20]:   # cap output at 20 rows
            print(json.dumps(r, default=str, indent=2))
    else:
        print(json.dumps(result, default=str, indent=2))

# ── Q1: Safest / least-safe cities and categories by avg stars ───────────────
print("\n--- Q1 ---")

q1_city = list(db.reviews.aggregate([
    {"$group": {
        "_id": "$business_id",
        "avg_stars": {"$avg": "$stars"},
        "review_count": {"$sum": 1}
    }},
    {"$match": {"review_count": {"$gte": 10}}},
    {"$lookup": {
        "from": "businesses",
        "localField": "_id",
        "foreignField": "_id",
        "as": "biz"
    }},
    {"$unwind": "$biz"},
    {"$group": {
        "_id": "$biz.city",
        "avg_stars": {"$avg": "$avg_stars"},
        "total_reviews": {"$sum": "$review_count"},
        "business_count": {"$sum": 1}
    }},
    {"$match": {"business_count": {"$gte": 5}}},
    {"$sort": {"avg_stars": -1}},
]))
pprint("Q1 - Top 5 safest cities", q1_city[:5])
pprint("Q1 - Top 5 least-safe cities", q1_city[-5:])

q1_cat = list(db.reviews.aggregate([
    {"$group": {
        "_id": "$business_id",
        "avg_stars": {"$avg": "$stars"},
        "review_count": {"$sum": 1}
    }},
    {"$match": {"review_count": {"$gte": 10}}},
    {"$lookup": {
        "from": "businesses",
        "localField": "_id",
        "foreignField": "_id",
        "as": "biz"
    }},
    {"$unwind": "$biz"},
    {"$unwind": "$biz.categories"},
    {"$group": {
        "_id": "$biz.categories",
        "avg_stars": {"$avg": "$avg_stars"},
        "total_reviews": {"$sum": "$review_count"},
        "business_count": {"$sum": 1}
    }},
    {"$match": {"business_count": {"$gte": 10}}},
    {"$sort": {"avg_stars": -1}},
]))
pprint("Q1 - Top 5 categories by avg stars", q1_cat[:5])
pprint("Q1 - Bottom 5 categories by avg stars", q1_cat[-5:])

# ── Q2: Strongest upward/downward trends by city and category ────────────────
print("\n--- Q2 ---")

q2_city = list(db.reviews.aggregate([
    {"$group": {
        "_id": {
            "business_id": "$business_id",
            "year": {"$year": "$date"}
        },
        "avg_stars": {"$avg": "$stars"}
    }},
    {"$lookup": {
        "from": "businesses",
        "localField": "_id.business_id",
        "foreignField": "_id",
        "as": "biz"
    }},
    {"$unwind": "$biz"},
    {"$group": {
        "_id": {"city": "$biz.city", "year": "$_id.year"},
        "avg_stars": {"$avg": "$avg_stars"}
    }},
    {"$sort": {"_id.city": 1, "_id.year": 1}},
]))
# Compute slope per city in Python
from collections import defaultdict
import statistics

city_years = defaultdict(list)
for r in q2_city:
    city_years[r["_id"]["city"]].append((r["_id"]["year"], r["avg_stars"]))

def slope(points):
    if len(points) < 2:
        return 0
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    n = len(xs)
    mx, my = sum(xs)/n, sum(ys)/n
    num = sum((x - mx)*(y - my) for x, y in zip(xs, ys))
    den = sum((x - mx)**2 for x in xs)
    return num / den if den else 0

city_slopes = [(c, slope(pts)) for c, pts in city_years.items() if len(pts) >= 3]
city_slopes.sort(key=lambda x: x[1], reverse=True)
pprint("Q2 - Cities with strongest upward trend (slope)", city_slopes[:5])
pprint("Q2 - Cities with strongest downward trend (slope)", city_slopes[-5:])

# ── Q3: Correlation between review volume and avg star rating ────────────────
print("\n--- Q3 ---")

q3 = list(db.reviews.aggregate([
    {"$group": {
        "_id": "$business_id",
        "avg_stars": {"$avg": "$stars"},
        "review_count": {"$sum": 1}
    }},
    {"$bucket": {
        "groupBy": "$review_count",
        "boundaries": [1, 10, 50, 100, 500, 1000, 10000],
        "default": "10000+",
        "output": {
            "avg_stars": {"$avg": "$avg_stars"},
            "business_count": {"$sum": 1}
        }
    }}
]))
pprint("Q3 - Avg stars by review count bucket", q3)

# ── Q4: Review behaviour by category ────────────────────────────────────────
print("\n--- Q4 ---")

q4 = list(db.reviews.aggregate([
    {"$lookup": {
        "from": "businesses",
        "localField": "business_id",
        "foreignField": "_id",
        "as": "biz"
    }},
    {"$unwind": "$biz"},
    {"$unwind": "$biz.categories"},
    {"$group": {
        "_id": "$biz.categories",
        "avg_stars": {"$avg": "$stars"},
        "avg_text_length": {"$avg": {"$strLenCP": "$text"}},
        "useful_per_review": {"$avg": "$useful"},
        "review_count": {"$sum": 1},
        "star_dist_1": {"$sum": {"$cond": [{"$eq": ["$stars", 1]}, 1, 0]}},
        "star_dist_2": {"$sum": {"$cond": [{"$eq": ["$stars", 2]}, 1, 0]}},
        "star_dist_3": {"$sum": {"$cond": [{"$eq": ["$stars", 3]}, 1, 0]}},
        "star_dist_4": {"$sum": {"$cond": [{"$eq": ["$stars", 4]}, 1, 0]}},
        "star_dist_5": {"$sum": {"$cond": [{"$eq": ["$stars", 5]}, 1, 0]}},
    }},
    {"$match": {"review_count": {"$gte": 200}}},
    {"$sort": {"review_count": -1}},
    {"$limit": 15}
]))
pprint("Q4 - Review behaviour by top categories", q4)

# ── Q5: Impact of user tenure on reviewing behaviour ─────────────────────────
print("\n--- Q5 ---")

q5 = list(db.reviews.aggregate([
    {"$lookup": {
        "from": "users",
        "localField": "user_id",
        "foreignField": "_id",
        "as": "user"
    }},
    {"$unwind": "$user"},
    {"$addFields": {
        "tenure_years": {
            "$divide": [
                {"$subtract": ["$date", {"$dateFromString": {"dateString": "$user.yelping_since"}}]},
                31536000000  # ms per year
            ]
        }
    }},
    {"$bucket": {
        "groupBy": "$tenure_years",
        "boundaries": [0, 1, 2, 3, 5, 8, 12, 20],
        "default": "20+",
        "output": {
            "avg_stars": {"$avg": "$stars"},
            "avg_useful": {"$avg": "$useful"},
            "review_count": {"$sum": 1}
        }
    }}
]))
pprint("Q5 - Reviewing behaviour by tenure", q5)

# ── Q6: Elite vs non-elite users ─────────────────────────────────────────────
print("\n--- Q6 ---")

q6 = list(db.reviews.aggregate([
    {"$lookup": {
        "from": "users",
        "localField": "user_id",
        "foreignField": "_id",
        "as": "user"
    }},
    {"$unwind": "$user"},
    {"$addFields": {
        "is_elite": {"$gt": [{"$size": {"$cond": [
          { "$isArray": "$user.elite" },
          "$user.elite",
          []
        ]}}, 0]}
    }},
    {"$group": {
        "_id": "$is_elite",
        "mean_stars": {"$avg": "$stars"},
        "mean_text_length": {"$avg": {"$strLenCP": "$text"}},
        "mean_useful": {"$avg": "$useful"},
        "review_count": {"$sum": 1}
    }}
]))
pprint("Q6 - Elite vs non-elite metrics", q6)

# ── Q7: Check-in activity vs star ratings ────────────────────────────────────
print("\n--- Q7 ---")

q7 = list(db.checkins.aggregate([
    {"$addFields": {
        "checkin_count": {
            "$size": {"$split": ["$date", ","]}
        }
    }},
    {"$lookup": {
        "from": "reviews",
        "localField": "business_id",
        "foreignField": "business_id",
        "as": "revs"
    }},
    {"$addFields": {
        "avg_stars": {"$avg": "$revs.stars"},
        "review_count": {"$size": "$revs"}
    }},
    {"$match": {"review_count": {"$gte": 5}}},
    {"$lookup": {
        "from": "businesses",
        "localField": "business_id",
        "foreignField": "_id",
        "as": "biz"
    }},
    {"$unwind": "$biz"},
    {"$unwind": "$biz.categories"},
    {"$group": {
        "_id": "$biz.categories",
        "avg_checkins": {"$avg": "$checkin_count"},
        "avg_stars": {"$avg": "$avg_stars"},
        "business_count": {"$sum": 1}
    }},
    {"$match": {"business_count": {"$gte": 10}}},
    {"$sort": {"avg_checkins": -1}},
    {"$limit": 20}
]))
pprint("Q7 - Checkin activity vs stars by category", q7)

print("\nAll queries done.")


--- Q1 ---

  Q1 - Top 5 safest cities
{
  "_id": "Valrico",
  "avg_stars": 4.4844841269841265,
  "total_reviews": 154,
  "business_count": 5
}
{
  "_id": "Seminole",
  "avg_stars": 4.170450382950383,
  "total_reviews": 172,
  "business_count": 7
}
{
  "_id": "Trenton",
  "avg_stars": 4.154141806128224,
  "total_reviews": 113,
  "business_count": 5
}
{
  "_id": "Newtown",
  "avg_stars": 4.148366831217077,
  "total_reviews": 345,
  "business_count": 7
}
{
  "_id": "Lutz",
  "avg_stars": 4.119673377697921,
  "total_reviews": 382,
  "business_count": 11
}

  Q1 - Top 5 least-safe cities
{
  "_id": "Plymouth Meeting",
  "avg_stars": 3.0009986138172176,
  "total_reviews": 305,
  "business_count": 8
}
{
  "_id": "Harvey",
  "avg_stars": 2.958210878946173,
  "total_reviews": 161,
  "business_count": 5
}
{
  "_id": "North Wales",
  "avg_stars": 2.8694023291584267,
  "total_reviews": 153,
  "business_count": 5
}
{
  "_id": "Downingtown",
  "avg_stars": 2.8684199134199133,
  "total_reviews": 14

In [3]:
import json
from neo4j import GraphDatabase

URI = "neo4j+s://15abb475.databases.neo4j.io"
AUTH = ("15abb475", "NuLUN5_RAMUQ3AQHtmG6w-0bftmavLct9VqGuJbVE5Y")

DATA_DIR = r"D:\Projects\DSM\Assignment2\Yelp JSON\yelp_dataset"

# Limits — keep small for Neo4j
BIZ_LIMIT    = None          # load all businesses (only ~150k, manageable)
USER_LIMIT   = 20_000
REVIEW_LIMIT = 80_000

driver = GraphDatabase.driver(URI, auth=AUTH)

def run(session, query, **kwargs):
    session.run(query, **kwargs)

# ── Helper to load jsonl ─────────────────────────────────────────────────────
def load_jsonl(path, limit=None):
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            yield json.loads(line)

with driver.session() as session:

    # ── Constraints ─────────────────────────────────────────────────────────
    print("Creating constraints...")
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (b:Business) REQUIRE b.business_id IS UNIQUE")
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE")
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE")
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (s:State) REQUIRE s.name IS UNIQUE")
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (r:Review) REQUIRE r.review_id IS UNIQUE")

    # ── Load Businesses + Categories ─────────────────────────────────────────
    print("Loading businesses...")
    batch = []
    for b in load_jsonl(f"{DATA_DIR}\\yelp_academic_dataset_business.json", BIZ_LIMIT):
        cats = [c.strip() for c in b.get("categories", "") .split(",")] if b.get("categories") else []
        batch.append({
            "business_id": b["business_id"],
            "name": b["name"],
            "city": b.get("city", ""),
            "state": b.get("state", ""),
            "stars": b.get("stars", 0),
            "review_count": b.get("review_count", 0),
            "is_open": b.get("is_open", 0),
            "categories": cats
        })
        if len(batch) == 2000:
            session.run("""
                UNWIND $batch AS b
                MERGE (biz:Business {business_id: b.business_id})
                SET biz.name = b.name,
                    biz.city = b.city,
                    biz.state = b.state,
                    biz.stars = b.stars,
                    biz.review_count = b.review_count,
                    biz.is_open = b.is_open
                WITH biz, b
                UNWIND b.categories AS catName
                MERGE (c:Category {name: catName})
                MERGE (biz)-[:IN_CATEGORY]->(c)
            """, batch=batch)
            batch = []
    if batch:
        session.run("""
            UNWIND $batch AS b
            MERGE (biz:Business {business_id: b.business_id})
            SET biz.name = b.name,
                biz.city = b.city,
                biz.state = b.state,
                biz.stars = b.stars,
                biz.review_count = b.review_count,
                biz.is_open = b.is_open
            WITH biz, b
            UNWIND b.categories AS catName
            MERGE (c:Category {name: catName})
            MERGE (biz)-[:IN_CATEGORY]->(c)
        """, batch=batch)
    print("  Businesses done.")

    # ── Load Users (subset) ─────────────────────────────────────────────────
    print("Loading users...")
    batch = []
    user_ids_loaded = set()
    for u in load_jsonl(f"{DATA_DIR}\\yelp_academic_dataset_user.json", USER_LIMIT):
        user_ids_loaded.add(u["user_id"])
        batch.append({
            "user_id": u["user_id"],
            "name": u.get("name", ""),
            "review_count": u.get("review_count", 0),
            "average_stars": u.get("average_stars", 0),
            "fans": u.get("fans", 0),
            "yelping_since": u.get("yelping_since", ""),
            "elite": u.get("elite", ""),
            "friends": [f.strip() for f in u.get("friends", "").split(",") if f.strip()][:50]  # cap friends
        })
        if len(batch) == 1000:
            session.run("""
                UNWIND $batch AS u
                MERGE (usr:User {user_id: u.user_id})
                SET usr.name = u.name,
                    usr.review_count = u.review_count,
                    usr.average_stars = u.average_stars,
                    usr.fans = u.fans,
                    usr.yelping_since = u.yelping_since,
                    usr.elite = u.elite
            """, batch=batch)
            batch = []
    if batch:
        session.run("""
            UNWIND $batch AS u
            MERGE (usr:User {user_id: u.user_id})
            SET usr.name = u.name,
                usr.review_count = u.review_count,
                usr.average_stars = u.average_stars,
                usr.fans = u.fans,
                usr.yelping_since = u.yelping_since,
                usr.elite = u.elite
        """, batch=batch)
    print("  Users done.")

    # ── Friend relationships (only within loaded users) ──────────────────────
    print("Creating FRIENDS relationships...")
    batch = []
    for u in load_jsonl(f"{DATA_DIR}\\yelp_academic_dataset_user.json", USER_LIMIT):
        friends = [f.strip() for f in u.get("friends", "").split(",") if f.strip()]
        for fid in friends[:20]:   # cap at 20 friends per user
            if fid in user_ids_loaded:
                batch.append({"uid": u["user_id"], "fid": fid})
        if len(batch) >= 5000:
            session.run("""
                UNWIND $batch AS rel
                MATCH (a:User {user_id: rel.uid})
                MATCH (b:User {user_id: rel.fid})
                MERGE (a)-[:FRIENDS_WITH]->(b)
            """, batch=batch)
            batch = []
    if batch:
        session.run("""
            UNWIND $batch AS rel
            MATCH (a:User {user_id: rel.uid})
            MATCH (b:User {user_id: rel.fid})
            MERGE (a)-[:FRIENDS_WITH]->(b)
        """, batch=batch)
    print("  Friend relationships done.")

    # ── Load Reviews ─────────────────────────────────────────────────────────
    print("Loading reviews...")
    batch = []
    for r in load_jsonl(f"{DATA_DIR}\\yelp_academic_dataset_review.json", REVIEW_LIMIT):
        batch.append({
            "review_id": r["review_id"],
            "user_id": r["user_id"],
            "business_id": r["business_id"],
            "stars": r.get("stars", 0),
            "date": r.get("date", ""),
            "useful": r.get("useful", 0)
        })
        if len(batch) == 2000:
            session.run("""
                UNWIND $batch AS r
                MATCH (u:User {user_id: r.user_id})
                MATCH (b:Business {business_id: r.business_id})
                MERGE (rev:Review {review_id: r.review_id})
                SET rev.stars = r.stars,
                    rev.date = r.date,
                    rev.useful = r.useful
                MERGE (u)-[:WROTE]->(rev)
                MERGE (rev)-[:REVIEWS]->(b)
            """, batch=batch)
            batch = []
    if batch:
        session.run("""
            UNWIND $batch AS r
            MATCH (u:User {user_id: r.user_id})
            MATCH (b:Business {business_id: r.business_id})
            MERGE (rev:Review {review_id: r.review_id})
            SET rev.stars = r.stars,
                rev.date = r.date,
                rev.useful = r.useful
            MERGE (u)-[:WROTE]->(rev)
            MERGE (rev)-[:REVIEWS]->(b)
        """, batch=batch)
    print("  Reviews done.")

driver.close()
print("\nNeo4j load complete.")

Creating constraints...
Loading businesses...


ClientError: {neo4j_code: Neo.ClientError.Transaction.TransactionHookFailed} {message: You have exceeded the logical size limit of 400000 relationships in your database (attempt to add 8826 relationships would reach 400449 relationships). Please consider upgrading to the next tier.} {gql_status: 50N00} {gql_status_description: error: general processing exception - internal error. Internal exception raised TransactionEventListeners: You have exceeded the logical size limit of 400000 relationships in your database (attempt to add 8826 relationships would reach 400449 relationships). Please consider upgrading to the next tier.}

In [ ]:
import json
from neo4j import GraphDatabase

URI = "neo4j+s://15abb475.databases.neo4j.io"
AUTH = ("15abb475", "NuLUN5_RAMUQ3AQHtmG6w-0bftmavLct9VqGuJbVE5Y")

def run_query(label, cypher, **params):
    with driver.session() as session:
        result = session.run(cypher, **params)
        records = [dict(r) for r in result]
        print(f"\n{'='*60}\n  {label}\n{'='*60}")
        for r in records[:20]:
            print(json.dumps(r, default=str, indent=2))
        return records

# ── N1: Top 10 users by friend count ─────────────────────────────────────────
run_query("N1 - Top 10 users by friend count",
"""
MATCH (u:User)-[:FRIENDS_WITH]->(f:User)
WITH u, COUNT(f) AS friend_count
ORDER BY friend_count DESC
LIMIT 10
MATCH (u)-[:WROTE]->(r:Review)
WITH u, friend_count, COUNT(r) AS total_reviews, AVG(r.stars) AS mean_stars
RETURN u.user_id AS user_id,
       u.name AS name,
       friend_count,
       total_reviews,
       round(mean_stars * 100) / 100.0 AS mean_stars
ORDER BY friend_count DESC
""")

# ── N2: Top 3 businesses per state by avg stars (min 50 reviews) ──────────────
run_query("N2 - Top 3 businesses per state by avg stars",
"""
MATCH (rev:Review)-[:REVIEWS]->(b:Business)
WITH b, AVG(rev.stars) AS avg_stars, COUNT(rev) AS review_count
WHERE review_count >= 50
WITH b.state AS state,
     b.name AS name,
     b.city AS city,
     avg_stars,
     review_count,
     [(b)-[:IN_CATEGORY]->(c) | c.name][0] AS category
ORDER BY state, avg_stars DESC
WITH state,
     COLLECT({name: name, city: city, avg_stars: avg_stars,
              review_count: review_count, category: category})[..3] AS top3
RETURN state, top3
ORDER BY state
""")

# ── N3: Top 10 users by distinct cities reviewed ──────────────────────────────
run_query("N3 - Top 10 users reviewed across most cities",
"""
MATCH (u:User)-[:WROTE]->(r:Review)-[:REVIEWS]->(b:Business)
WITH u, b.city AS city, AVG(r.stars) AS city_stars
WITH u, COUNT(DISTINCT city) AS cities_covered,
     COLLECT({city: city, avg_stars: city_stars}) AS city_stats
ORDER BY cities_covered DESC
LIMIT 10
RETURN u.user_id AS user_id,
       u.name AS name,
       cities_covered,
       city_stats
""")

# ── N4: Category-specific vs overall mean stars ───────────────────────────────
CATEGORY = "Restaurants"   # change if you want a different one
run_query(f"N4 - Users who reviewed 5+ {CATEGORY} businesses: category vs overall avg",
f"""
MATCH (u:User)-[:WROTE]->(r:Review)-[:REVIEWS]->(b:Business)-[:IN_CATEGORY]->(c:Category)
WHERE c.name = $cat
WITH u, COUNT(DISTINCT b) AS biz_count, AVG(r.stars) AS cat_avg
WHERE biz_count >= 5
MATCH (u)-[:WROTE]->(allR:Review)
WITH u, biz_count, cat_avg, AVG(allR.stars) AS overall_avg
RETURN u.user_id AS user_id,
       u.name AS name,
       biz_count,
       round(cat_avg * 100) / 100.0 AS category_avg_stars,
       round(overall_avg * 100) / 100.0 AS overall_avg_stars,
       round((cat_avg - overall_avg) * 100) / 100.0 AS difference
ORDER BY biz_count DESC
LIMIT 20
""", cat=CATEGORY)

# ── N5: Reproduce MongoDB Q6 (elite vs non-elite) in Cypher ──────────────────
run_query("N5 - Elite vs non-elite (reproducing Q6 in Cypher)",
"""
MATCH (u:User)-[:WROTE]->(r:Review)
WITH u,
     CASE WHEN u.elite IS NOT NULL AND u.elite <> '' AND u.elite <> 'None'
          THEN true ELSE false END AS is_elite,
     r.stars AS stars
WITH is_elite,
     COUNT(*) AS review_count,
     AVG(stars) AS mean_stars
RETURN is_elite, review_count, round(mean_stars * 100) / 100.0 AS mean_stars
ORDER BY is_elite DESC
""")

driver.close()
print("\nNeo4j queries done.")